In [106]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import pandas as pd

In [28]:
# Define feature columns
tech_cols = list(group_tech_matrix_df.columns)

feature_cols = ['year', 'month', 'industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type'] + tech_cols

# Identify Categorical vs Numeric
categorical_cols = ['industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type']
numeric_cols =['year', 'month'] + tech_cols

In [31]:
# Preprocess with one-hot encoding (fit on all data)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# Fit preprocessor on full dataset
preprocessor.fit(events_plus_matrix_df[feature_cols])

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [32]:
# Filter out NaN values for the respective training sets
train_df_origin = events_plus_matrix_df[events_plus_matrix_df['origin_risk_norm'].notna()]
train_df_victim = events_plus_matrix_df[events_plus_matrix_df['victim_risk_norm'].notna()]

In [33]:
# Target Variables (y)
y_origin = train_df_origin['origin_risk_norm']
y_victim = train_df_victim['victim_risk_norm']

# Input Features (X)
X_origin_raw = train_df_origin[feature_cols]
X_origin = preprocessor.transform(X_origin_raw)
X_victim_raw = train_df_victim[feature_cols]
X_victim = preprocessor.transform(X_victim_raw)

In [34]:
# Train models
origin_model = GradientBoostingRegressor()
victim_model = GradientBoostingRegressor()

origin_model.fit(X_origin, y_origin)
victim_model.fit(X_victim, y_victim)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [36]:
# Predict for all events
X_all = preprocessor.transform(events_plus_matrix_df[feature_cols])

events_plus_matrix_df['ml_origin_risk'] = origin_model.predict(X_all)
events_plus_matrix_df['ml_target_risk'] = victim_model.predict(X_all)

In [41]:
#events_df
#events_plus_matrix_df

## Extrapolating to the Other APTs

In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

# Group technique names by APT group_name
grouped = (
    group_techniques_df
    .groupby("group_name")["technique_name"]
    .apply(list)
)

# Multi-label binarize
mlb = MultiLabelBinarizer()
tech_matrix = mlb.fit_transform(grouped)

# Build the technique matrix DataFrame
group_tech_matrix_df = pd.DataFrame(
    tech_matrix,
    index=grouped.index,      # IMPORTANT: use .index, not grouped directly
    columns=mlb.classes_
)

group_tech_matrix_df.tail(10)

,ARP Cache Poisoning,Abuse Elevation Control Mechanism,Access Token Manipulation,Accessibility Features,Account Access Removal,Account Discovery,Account Manipulation,Acquire Access,Acquire Infrastructure,Add-ins,...,Windows File and Directory Permissions Modification,Windows Management Instrumentation,Windows Management Instrumentation Event Subscription,Windows Remote Management,Windows Service,Winlogon Helper DLL,Wordlist Scanning,XDG Autostart Entries,XSL Script Processing,vSphere Installation Bundles
group_name,,,,,,,,,,,,,,,,,,,,,
Water Galura,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Whitefly,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Windigo,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Windshift,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
Winnti Group,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Winter Vivern,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Wizard Spider,0,0,0,0,0,0,0,0,0,0,...,1,1,0,1,1,1,0,0,0,0
ZIRCONIUM,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
admin@338,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
# Set up a new dataframe to house the tech matrix too
events_df["apt_group_mitre"] = events_df["apt_group"].map(alias_to_official)

events_plus_matrix_df = events_df.merge(
    group_tech_matrix_df,
    left_on="apt_group_mitre",
    right_index=True,
    how="left"
)

# Replace NaNs for events with no APT attribution
events_plus_matrix_df[group_tech_matrix_df.columns] = events_plus_matrix_df[group_tech_matrix_df.columns].fillna(0)

In [43]:
cols_to_view = ['slug',
 'original_method',
 'event_date',
 'reported_date',
 'year',
 'month',
 'actor',
 'actor_type',
 'organization',
 'industry_code',
 'industry',
 'motive',
 'event_type',
 'event_subtype',
 'magnitude',
 'duration',
 'scope',
 'ip',
 'org_data',
 'cust_data',
 'description',
 'source_url',
 'country',
 'actor_country',
 'state',
 'county',
 'change_log',
 'apt_group',
 'origin_risk_norm',
 'victim_risk_norm',
 'ml_origin_risk',
 'ml_target_risk',
 'apt_group_mitre']

In [49]:
apt_events = events_plus_matrix_df[
    events_plus_matrix_df["apt_group_mitre"].notna()
]

apt_events[cols_to_view]


,slug,original_method,event_date,reported_date,year,month,actor,actor_type,organization,industry_code,...,actor_country,state,county,change_log,apt_group,origin_risk_norm,victim_risk_norm,ml_origin_risk,ml_target_risk,apt_group_mitre
358,51de7a6f8a84858e,1,2014-06-30,NaN,2014,6,Green Dragon Crew,Hacktivist,Privatbank,52,...,Ukraine,NaN,NaN,NaN,dragonok,0.680573,0.679409,0.482337,0.536513,DragonOK
786,2c0603efa7d7541b,1,2015-02-28,NaN,2015,2,f AKA @Cleaver,Undetermined,frontalot.com,81,...,Undetermined,Undetermined,Undetermined,NaN,cleaver,NaN,1.000000,0.487850,0.745479,Cleaver
789,1147a5bf27ee80f6,1,2015-03-01,NaN,2015,3,Ministry of State Security's (MSS) Guangdong S...,Nation-State,Hanford Site,92,...,China,Washington,Benton,NaN,apt3,0.718757,1.000000,0.754085,0.743916,APT3
1005,649f14c0fd38146e,1,2015-05-31,NaN,2015,5,APT32,Nation-State,Armed Forces of the Philippines,92,...,Viet Nam,NaN,NaN,NaN,apt32,0.457705,0.496255,0.476694,0.479695,APT32
1006,c786e72aab5694ab,1,2015-05-31,NaN,2015,5,APT32,Nation-State,ASEAN Investment,92,...,Viet Nam,NaN,NaN,NaN,apt32,0.457705,NaN,0.476694,0.479695,APT32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15748,z1r8m5x2c9v4k7t3,1,2025-03-25,2025-10-23,2025,3,NGB 3rd Technical Surveillance Bureau (Lazarus...,Nation-state,Undisclosed European drone manufacturer,31,...,Korea (the Democratic People's Republic of),Undetermined,Undetermined,NaN,lazarus group,0.691554,NaN,0.637910,0.630050,Lazarus Group
15755,q6v2r9x1c7m5t3k8,1,2025-09-01,2025-10-24,2025,9,APT36 (Transparent Tribe),Nation-state,Undisclosed Indian government or infrastructur...,92,...,Pakistan,NaN,NaN,NaN,transparent tribe,0.551293,0.661710,0.594307,0.613004,Transparent Tribe
15758,z7m3t1p9v4x2k8r5,1,2025-07-15,2025-10-21,2025,7,Ministry of State Security's (MSS) (Salt Typhoon),Nation-state,Undisclosed European telecommunications company,51,...,China,Undetermined,Undetermined,NaN,salt typhoon,0.718757,NaN,0.730326,0.872664,Salt Typhoon
15770,b9e1f4d2c7a3e805,1,2025-09-01,2025-10-30,2025,9,UNC6384,Nation-State,Undisclosed Belgian Diplomatic Entities,92,...,China,NaN,NaN,NaN,mustang panda,0.718757,0.487955,0.667285,0.438859,Mustang Panda


In [50]:
default_metadata = {
    'year': 2025,
    'month': 1,
    'industry_code': 'Unknown',
    'event_type': 'Unknown',
    'event_subtype': 'Unknown',
    'motive': 'Unknown',
    'actor_type': 'Unknown'
}

In [60]:
def build_apt_features(
    group_name,
    group_tech_matrix_df=group_tech_matrix_df,
    default_metadata=default_metadata
):
    # Technique vector (multi-hot row)
    ttp_vec = group_tech_matrix_df.loc[[group_name]]

    # Metadata row
    meta_df = pd.DataFrame([default_metadata])

    # Combine them
    combined = pd.concat([meta_df, ttp_vec.reset_index(drop=True)], axis=1)
    return combined


In [ ]:
#build_apt_features("Indrik Spider")

,year,month,industry_code,event_type,event_subtype,motive,actor_type,ARP Cache Poisoning,Abuse Elevation Control Mechanism,Access Token Manipulation,...,Windows File and Directory Permissions Modification,Windows Management Instrumentation,Windows Management Instrumentation Event Subscription,Windows Remote Management,Windows Service,Winlogon Helper DLL,Wordlist Scanning,XDG Autostart Entries,XSL Script Processing,vSphere Installation Bundles
0,2025,1,Unknown,Unknown,Unknown,Unknown,Unknown,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [66]:
def score_all_mitre_apts(
    groups_df,
    events_df,
    group_tech_matrix_df,
    preprocessor,
    origin_model,
    victim_model,
    default_metadata
):
    results = []

    # All MITRE APT names
    mitre_groups = groups_df["name"].unique()

    for group_name in mitre_groups:

        # -----------------------------------------
        # 1. CHECK IF APT APPEARS IN UMD EVENTS DF
        # -----------------------------------------
        match_rows = events_df[events_df["apt_group_mitre"] == group_name]

        has_ml_scores = (
            "ml_origin_risk" in match_rows
            and "ml_target_risk" in match_rows
            and not match_rows["ml_origin_risk"].isna().all()
            and not match_rows["ml_target_risk"].isna().all()
        )

        if has_ml_scores:
            # Use the *ML* values already predicted in events_df
            ml_origin = match_rows["ml_origin_risk"].mean()
            ml_target = match_rows["ml_target_risk"].mean()

            results.append({
                "apt_group": group_name,
                "origin_risk": ml_origin,
                "target_risk": ml_target,
                "source": "ml_from_events"
            })
            continue


        # ---------------------------------------------------
        # 2. IF NOT IN UMD → USE METHOD A ML PREDICTION
        # ---------------------------------------------------
        if group_name in group_tech_matrix_df.index:

            # Get technique multi-hot vector
            ttp_vec = group_tech_matrix_df.loc[[group_name]]

            # Default metadata
            meta_df = pd.DataFrame([default_metadata])

            # Build the model input row
            model_input = pd.concat(
                [meta_df, ttp_vec.reset_index(drop=True)], axis=1
            )

            # Predict using the trained ML models
            X = preprocessor.transform(model_input)
            ml_origin = float(origin_model.predict(X)[0])
            ml_target = float(victim_model.predict(X)[0])

            results.append({
                "apt_group": group_name,
                "origin_risk": ml_origin,
                "target_risk": ml_target,
                "source": "ml_from_mitre"
            })

        else:
            # ------------------------------------------------
            # 3. NO TECHNIQUES + NO EVENTS → Cannot score it
            # ------------------------------------------------
            results.append({
                "apt_group": group_name,
                "origin_risk": None,
                "target_risk": None,
                "source": "no_data"
            })

    return pd.DataFrame(results)


In [69]:
apt_scores_df = score_all_mitre_apts(
    groups_df=groups_df,
    events_df=events_plus_matrix_df,        # includes ml_origin_risk, ml_target_risk
    group_tech_matrix_df=group_tech_matrix_df,
    preprocessor=preprocessor,
    origin_model=origin_model,
    victim_model=victim_model,
    default_metadata=default_metadata
)

apt_scores_df

,apt_group,origin_risk,target_risk,source
0,Indrik Spider,0.787988,0.702124,ml_from_mitre
1,LuminousMoth,0.761883,0.621181,ml_from_mitre
2,Medusa Group,0.765589,0.562167,ml_from_mitre
3,Wizard Spider,0.804625,0.625153,ml_from_mitre
4,Elderwood,0.761404,0.616799,ml_from_mitre
...,...,...,...,...
182,Taidoor,NaN,NaN,no_data
183,APT-C-23,0.717367,0.672121,ml_from_events
184,MONSOON,NaN,NaN,no_data
185,Charming Kitten,0.704467,0.662877,ml_from_events


## Process the Final Data for Export

In [103]:
final_apt_scores_df = apt_scores_df[apt_scores_df["source"] != "no_data"].copy()

# Normalize the scores
max_origin = final_apt_scores_df["origin_risk"].max()
final_apt_scores_df["origin_risk_norm"] = (
    final_apt_scores_df["origin_risk"] / max_origin
)

max_target = final_apt_scores_df["target_risk"].max()
final_apt_scores_df["target_risk_norm"] = (
    final_apt_scores_df["target_risk"] / max_target
)

# Split the dfs
origin_df = final_apt_scores_df[["apt_group", "origin_risk_norm"]].copy()
target_df = final_apt_scores_df[["apt_group", "target_risk_norm"]].copy()

# Sort the dfs
origin_df = origin_df.sort_values(by="origin_risk_norm", ascending=False)
target_df = target_df.sort_values(by="target_risk_norm", ascending=False)

In [ ]:
#origin_df

,apt_group,origin_risk_norm
44,Magic Hound,1.000000
112,BlackByte,0.928090
175,AppleJeus,0.906690
65,LAPSUS$,0.904582
117,Sea Turtle,0.897409
...,...,...
54,APT32,0.464808
146,Cleaver,0.460437
181,DragonOK,0.455234
152,Carbanak,0.429932


In [ ]:
#target_df

,apt_group,target_risk_norm
96,Scattered Spider,1.000000
152,Carbanak,0.956248
6,FIN7,0.954720
18,TA505,0.953506
121,Salt Typhoon,0.934579
...,...,...
64,Mustang Panda,0.587193
111,Confucius,0.583503
45,APT29,0.581709
54,APT32,0.567073


In [ ]:
# Export the dataframes as CSVs
origin_export = origin_df.rename(columns={
    "apt_group": "group_name",
    "origin_risk_norm": "score"
})[["group_name", "score"]]

target_export = target_df.rename(columns={
    "apt_group": "group_name",
    "target_risk_norm": "score"
})[["group_name", "score"]]

origin_export.to_csv(APT_ROOT / "analysis_data/origin.csv", index=False)
target_export.to_csv(APT_ROOT / "analysis_data/target.csv", index=False)